In [3]:
!pip install -q peft transformers datasets accelerate torch

In [4]:
from datasets import Dataset

data = [
    {"text": "0x5Fbe sent 200,000 BTW to 0x1Ae on May 4 2026", "label": '{"token": "BTW", "amount": 200000, "from": "0x5Fbe", "to": "0x1Ae", "date": "2026-05-04"}'},

]

# For demo,
import random
tokens = ["BTW", "USDC", "ETH", "SOL"]
dates = ["May 1 2026", "Apr 28 2026", "May 5 2026"]
for i in range(28):
    t = random.choice(tokens)
    amt = random.randint(1000, 500000)
    from_ = f"0x{random.randint(1,999):03x}"
    to_ = f"0x{random.randint(1,999):03x}"
    date = random.choice(dates)
    text = f"{from_} sent {amt} {t} to {to_} on {date}"
    label = f'{{"token": "{t}", "amount": {amt}, "from": "{from_}", "to": "{to_}", "date": "{date}"}}'
    data.append({"text": text, "label": label})

dataset = Dataset.from_list(data)
dataset = dataset.train_test_split(test_size=0.2)

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [5]:
!pip uninstall peft -y
!pip install peft==0.10.0

Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 5.0 MB/s eta 0:00:00


In [15]:
def preprocess_function(examples):
    # Tokenize inputs
    inputs = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)
    # Tokenize labels as target text
    labels = tokenizer(text_target=examples["label"], truncation=True, padding="max_length", max_length=128)
    inputs["labels"] = labels["input_ids"]
    return inputs

# Apply mapping and remove original columns to avoid conflict
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/23 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

In [16]:
print(tokenized_dataset["train"][0]["labels"])

[3, 2, 121, 235, 2217, 121, 10, 96, 27333, 1686, 96, 9, 11231, 121, 10, 6352, 4906, 2668, 6, 96, 7152, 121, 10, 96, 632, 226, 10124, 1686, 96, 235, 121, 10, 96, 632, 226, 519, 75, 75, 1686, 96, 5522, 121, 10, 96, 188, 102, 52, 2059, 460, 2688, 121, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [12]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q", "v"]
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 344,064 || all params: 77,305,216 || trainable%: 0.445072166928555


In [18]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./output",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    logging_steps=5,
    save_strategy="no",
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],   # still passed but won't trigger automatic eval
)
trainer.train()

Step,Training Loss
5,11.048571
10,11.067664
15,11.009312
20,11.025999
25,11.008826
30,11.010374
35,11.002762
40,10.988501
45,11.008245
50,10.982351


TrainOutput(global_step=60, training_loss=11.009805234273275, metrics={'train_runtime': 227.3119, 'train_samples_per_second': 1.012, 'train_steps_per_second': 0.264, 'total_flos': 10749479485440.0, 'train_loss': 11.009805234273275, 'epoch': 10.0})

In [19]:
test_input = "0xabc sent 12345 BTW to 0xdef on May 6 2026"
inputs = tokenizer(test_input, return_tensors="pt")
outputs = peft_model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

0xabc ne 12345
